# Notebook 3 — supplément (sécurité, biais, XAI)

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import accuracy_score, f1_score
from src import config, data, preprocessing, models, fairness, attack
from src.models import regression_metrics, metrics_to_df
sns.set_theme(style='whitegrid')
config.FIG_DIR.mkdir(parents=True, exist_ok=True)
config.RES_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = config.RANDOM_STATE


## Réentraînement HistGBRT

Sous-échantillon 100k.

In [2]:
from src import features
df = data.load_train(nrows=200_000)
df = features.add_derived(df)
df = data.add_at_risk_label(df)
X_train, X_val, X_test, y_train, y_val, y_test = data.make_splits(df, features=config.FEATURES_FINAL_PLUS)
pipe = models.make_sklearn_models()['hist_gbrt']
pipe.fit(X_train, y_train)
pred_test = pipe.predict(X_test)
print('test RMSE:', np.sqrt(((pred_test - y_test)**2).mean()).round(3))


test RMSE: 9.181


## Sécurité — attaque d'évasion

Un étudiant peut-il mentir un peu pour passer en at_risk ?

In [3]:
# Cas concret : un étudiant initialement prédit OK
row = X_test.iloc[0].copy()
from src.attack import evade
res = evade(pipe, row)
print('prédiction sincère :', round(res.base_pred, 2))
print('prédiction après attaque :', round(res.attack_pred, 2))
print('changements :', res.changes)
print('succès :', res.success, '| nb changements :', res.n_features_changed)


prédiction sincère : 87.44
prédiction après attaque : 87.44
changements : {}
succès : False | nb changements : 0


In [4]:
# Évaluation systématique sur 200 étudiants initialement OK
stats = attack.evaluate_attack_success(pipe, X_test, n=200)
print(json.dumps(stats, indent=2))
with open(config.RES_DIR / 'attack_summary.json','w') as f:
    json.dump(stats, f, indent=2)


{
  "n_evaluated": 200,
  "n_success": 107,
  "success_rate": 0.535,
  "mean_changes_on_success": 1.2242990654205608
}


Mitigation : recoupement avec données objectives (présence, notes intermédiaires).

## Biais — équité par genre

Modèle avec et sans la variable genre.

In [5]:
# Métriques d'équité du modèle complet
ftab_full = fairness.fairness_table(y_test, pred_test, X_test['genre'])
print('=== Modèle complet ===')
ftab_full


=== Modèle complet ===


,group,n,selection_rate(at_risk),accuracy,TPR(recall_at_risk),FPR,MAE_score,RMSE_score
0,female,12548,0.208,0.885,0.705,0.062,7.208,9.079
1,male,17452,0.265,0.872,0.748,0.081,7.345,9.254


In [6]:
# Modèle sans `genre` (mitigation par retrait simple) - mêmes hyperparams
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingRegressor
cat_no_genre = [c for c in config.CAT_COLS_KEPT if c != 'genre']
num_with_derived = config.NUM_COLS_KEPT + config.DERIVED_COLS
pre_no_g = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median'))]), num_with_derived),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='missing')),
                      ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_no_genre),
], remainder='drop', verbose_feature_names_out=False)
pipe_no_g = Pipeline([('pre', pre_no_g),
    ('model', HistGradientBoostingRegressor(max_depth=8, learning_rate=0.06,
                                            max_iter=400, min_samples_leaf=80,
                                            l2_regularization=1e-3, random_state=RANDOM_STATE))])
X_train2 = X_train.drop(columns=['genre']); X_test2 = X_test.drop(columns=['genre'])
pipe_no_g.fit(X_train2, y_train)
pred_test_ng = pipe_no_g.predict(X_test2)
ftab_no_g = fairness.fairness_table(y_test, pred_test_ng, X_test['genre'])
print('=== Modèle SANS genre ===')
ftab_no_g


=== Modèle SANS genre ===


,group,n,selection_rate(at_risk),accuracy,TPR(recall_at_risk),FPR,MAE_score,RMSE_score
0,female,12548,0.236,0.883,0.762,0.081,7.343,9.219
1,male,17452,0.243,0.872,0.709,0.066,7.381,9.314


In [7]:
# Comparaison performance vs équité
from sklearn.metrics import mean_squared_error
rmse_full = mean_squared_error(y_test, pred_test) ** 0.5
rmse_ng   = mean_squared_error(y_test, pred_test_ng) ** 0.5
summary = pd.DataFrame({
    'modèle': ['avec genre', 'sans genre'],
    'RMSE test': [rmse_full, rmse_ng],
    'écart TPR (max-min)': [fairness.disparity(ftab_full,'TPR(recall_at_risk)'),
                            fairness.disparity(ftab_no_g,'TPR(recall_at_risk)')],
    'écart selection_rate': [fairness.disparity(ftab_full,'selection_rate(at_risk)'),
                             fairness.disparity(ftab_no_g,'selection_rate(at_risk)')],
}).round(4)
summary.to_csv(config.RES_DIR / 'fairness_summary.csv', index=False)
summary


,modèle,RMSE test,écart TPR (max-min),écart selection_rate
0,avec genre,9.1813,0.043,0.057
1,sans genre,9.2745,0.053,0.007


Retirer genre réduit peu la disparité (proxies via méthode_etude). Mitigations plus fortes : reweighting, post-proc.

## XAI

Permutation + PDP du notebook 1 répondent déjà. Pour aller plus loin : SHAP, contrefactuels.

In [8]:
# Exemple d'explication 'locale' textuelle pour un étudiant à risque
from src.attack import evade
candidates = X_test[pred_test < 50].head(3)
for i, (_, row) in enumerate(candidates.iterrows()):
    pred = float(pipe.predict(pd.DataFrame([row]))[0])
    print(f'\n=== Étudiant #{i+1} — note prédite = {pred:.1f} ===')
    print('Features renseignées :', dict(row))



=== Étudiant #1 — note prédite = 42.8 ===
Features renseignées : {'heures_etude': 0.08, 'assiduité_classe': 93.8, 'heures_sommeil': 5.0, 'study_efficiency': 0.0, 'sleep_score': 3.0, 'risk_internet_low': 0, 'genre': 'male', 'diplôme': 'Engineering', 'accès_internet': 'yes', 'qualité_sommeil': 'average', 'méthode_etude': 'self-study', 'évaluation_établissement': 'high'}

=== Étudiant #2 — note prédite = 32.5 ===
Features renseignées : {'heures_etude': 0.19, 'assiduité_classe': 49.9, 'heures_sommeil': 7.5, 'study_efficiency': 0.0, 'sleep_score': 2.25, 'risk_internet_low': 0, 'genre': 'male', 'diplôme': 'Engineering', 'accès_internet': 'yes', 'qualité_sommeil': 'poor', 'méthode_etude': 'group study', 'évaluation_établissement': 'high'}

=== Étudiant #3 — note prédite = 32.7 ===
Features renseignées : {'heures_etude': 1.21, 'assiduité_classe': 74.7, 'heures_sommeil': 5.2, 'study_efficiency': 0.0, 'sleep_score': 1.56, 'risk_internet_low': 0, 'genre': 'male', 'diplôme': 'Business Management'